In [1]:
perfil = "SRD1CIV"
tipo_processo = "EXECUÇÃO FISCAL"

In [2]:
# Importa tudo, loga, entra no perfil

from selenium import webdriver
from selenium.webdriver.support.select import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.alert import Alert
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import ElementClickInterceptedException

#Bibliotecas de Sistema
import time
import re
import csv
import os
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd
from contextlib import closing
from sympy import false

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama

#pasta_downloads = r"C:\Users\dodonin\Downloads"
pasta_downloads = r"D:\Downloads"

navegador = eproc.novo_browser(pasta_downloads)

#configura variáveis
username = "dodonin"
password = keyring.get_password("eproc", username)
pyotop_code = "GJRGIYTCGBSGKYTEHE2TOZRUGFQTMMRQ"

#eproc.login_no_eproc_tj(navegador, username, password, pyotop_code)
eproc.login_no_eproc(navegador, username, password, pyotop_code)

# Entra no perfil da Vara
eproc.entrar_no_perfil(navegador, perfil)

Driver do Eproc importado
Perfil carregado: SRD1CIV


In [3]:
#Pega os processos novos para minutar, adiciona coluna de dias, pagina por 100

# Espera até o elemento estar presente e clicável
meus_localizadores = WebDriverWait(navegador, 20).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'i[title="Meus Localizadores"]'))
)
meus_localizadores.click()

# Aguarda algum carregamento após o clique, se necessário (exemplo: espera um painel aparecer)
# WebDriverWait(navegador, 20).until(
#     EC.visibility_of_element_located((By.ID, "id_do_painel_ou_elemento_esperado"))
# )

# Localiza o primeiro <td> que contenha "CÍVEL - MINUTAR" no texto
td_civel_minutar = WebDriverWait(navegador, 20).until(
    EC.presence_of_element_located((By.XPATH, '//td[contains(text(), "CÍVEL - MINUTAR")]'))
)

# Encontra o <td> imediatamente a seguir
td_seguinte = td_civel_minutar.find_element(By.XPATH, 'following-sibling::td[1]')

# Dentro desse <td>, localiza o <a> e clica, esperando estar clicável
a_element = WebDriverWait(td_seguinte, 20).until(
    EC.element_to_be_clickable((By.TAG_NAME, 'a'))
)
a_element.click()

# Localiza e clica no label "100 processos por página"
label_100 = WebDriverWait(navegador, 20).until(
    EC.element_to_be_clickable((By.XPATH, '//label[contains(text(), "100 processos por página")]'))
)
label_100.click()

# Localiza o label com id "lbloptNdiasSituacao" e clica apenas se o checkbox estiver desmarcado
label_ndias_situacao = WebDriverWait(navegador, 20).until(
    EC.element_to_be_clickable((By.ID, "lbloptNdiasSituacao"))
)
checkbox_ndias = navegador.find_element(By.ID, "optNdiasSituacao")
if not checkbox_ndias.is_selected():
    label_ndias_situacao.click()

# Localiza o botão com id "btnConsultar" e exibe na tela
botao_consultar = navegador.find_element(By.ID, "btnConsultar")
navegador.execute_script("arguments[0].scrollIntoView();", botao_consultar)

# Tenta clicar no botão "Consultar", rolando para garantir visibilidade e tratando possíveis interceptações

try:
    botao_consultar.click()
except ElementClickInterceptedException:
    navegador.execute_script("arguments[0].scrollIntoView({block: 'center'});", botao_consultar)
    time.sleep(1)
    botao_consultar.click()

In [4]:
#Define funções para pegar os dados da tabela


from pydoc import text
from sympy import true


def pega_tabela_pagina(dados_tabela):
    tabela = navegador.find_element(By.ID, "tabelaLocalizadores")
    linhas = tabela.find_elements(By.TAG_NAME, "tr")[1:]  # Ignora o cabeçalho
    lastpage = false
    while lastpage == false:
        for linha in linhas:
            # Aguarda o carregamento do tbody da tabela antes de processar as linhas
            WebDriverWait(navegador, 10).until(
                EC.presence_of_element_located((By.XPATH, "//table[@id='tabelaLocalizadores']/tbody"))
            )
            colunas = linha.find_elements(By.TAG_NAME, "td")[:4]
            if len(colunas) >= 3:
                # Adiciona apenas a primeira linha, sem quebra de linha
                dados_tabela.append([colunas[1].text.split('\n')[0], colunas[2].text.split('\n')[0], colunas[3].text.split('\n')[0]])
            
            lastpage = true

def pega_ultima_peticao(navegador):
    documentos_eventos = []
    eventos = navegador.find_elements(By.CLASS_NAME, "td-evento")

    for evento in eventos:
        doc_id = evento.get_dom_attribute("id")
        tr_element = evento.find_element(By.XPATH, "./ancestor::tr")
        evento_id = tr_element.find_element(By.XPATH, './td[2]').text
        # Garante que evento_id seja apenas um int (remove qualquer caractere não numérico)
        evento_id = ''.join(filter(str.isdigit, evento_id))
        
        # Pega o atributo data-nome do elemento com classe infraLinkDocumento dentro do link
        try:
            infra_link = evento.find_element(By.XPATH, ".//*[contains(@class, 'infraLinkDoc')]")
            data_nome = infra_link.get_attribute("data-nome")
        except Exception:
            data_nome = None

        documentos_eventos.append((doc_id, data_nome, evento_id))

    documentos_eventos_filtrados = [item for item in documentos_eventos if item[1] == "PET"]

    documento_requerido = documentos_eventos_filtrados[0]
    texto_peticao = eproc.pega_texto_documento(navegador, documento_requerido[0])

    return texto_peticao

def ollama_resumo(pedido):
    print("Iniciando resumo com LLM...")
    pergunta_gemma = "Considere o seguinte pedido." \
    f"{pedido}" \
    "Resuma, da maneira mais objetiva possível, o pedido. Não mencione dados pessoais, como nomes, números de documento, números de processo, valores, etc. " \
    "O resumo deve ser genérico e breve (uma frase apenas, com o mínimo de palavras possível). " \
    "Se tiver mais de um pedido, retorne uma frase para cada um." \

    resumo = ollama.chat(
        model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
        messages=[{'role': 'user', 'content': f'{pergunta_gemma}'}],    
    )

    return(resumo['message']['content'])


In [5]:
#Limpa a tabela do perfil
with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()
    cursor.execute(f"DELETE FROM {perfil}")
    conn.commit()

In [6]:
# Localiza a tabela pelo id e importa as três primeiras colunas (ignorando o cabeçalho)
dados_tabela = []

pega_tabela_pagina(dados_tabela)

print(dados_tabela)

table_name = f"{perfil}"

with sqlite3.connect("urcaciv.db") as conn:
    cursor = conn.cursor()

    cursor.executemany(
        f"INSERT INTO {table_name} (num_processo, dias, tipo) VALUES (?, ?, ?)",
        dados_tabela
    )
    conn.commit()

[['5009747-60.2024.8.21.0009', '1', 'OUTROS PROCEDIMENTOS DE JURISDIÇÃO VOLUNTÁRIA'], ['5009541-70.2021.8.21.0132', '9', 'Guarda'], ['5007601-17.2022.8.21.0009 ', '16', 'PROCEDIMENTO COMUM CÍVEL'], ['5006577-82.2020.8.21.0086 ', '10', 'Interdição/Curatela'], ['5004747-45.2025.8.21.0009', '2', 'CUMPRIMENTO DE SENTENÇA DE OBRIGAÇÃO DE PRESTAR ALIMENTOS'], ['5004547-86.2024.8.21.0069', '1', 'OUTROS PROCEDIMENTOS DE JURISDIÇÃO VOLUNTÁRIA'], ['5004537-42.2024.8.21.0069', '15', 'PROCEDIMENTO COMUM CÍVEL'], ['5004529-65.2024.8.21.0069', '9', 'PROCEDIMENTO COMUM CÍVEL'], ['5004525-28.2024.8.21.0069 ', '34', 'MANDADO DE SEGURANÇA'], ['5004481-43.2023.8.21.0069', '8', 'EXECUÇÃO FISCAL'], ['5004480-58.2023.8.21.0069', '48', 'EXECUÇÃO FISCAL'], ['5004479-73.2023.8.21.0069', '17', 'EXECUÇÃO FISCAL'], ['5004469-92.2024.8.21.0069', '7', 'CUMPRIMENTO DE SENTENÇA DE OBRIGAÇÃO DE PRESTAR ALIMENTOS'], ['5004467-25.2024.8.21.0069', '23', 'ALIMENTOS - LEI ESPECIAL Nº 5.478/68'], ['5004462-03.2024.8.21.0069

In [7]:
#guarda os processos no db
with sqlite3.connect("urcaciv.db") as conn:
    df_pendentes = pd.read_sql_query(f"SELECT * FROM {perfil} WHERE tipo = '{tipo_processo}'", conn)
df_pendentes = df_pendentes["num_processo"].tolist()
print("processos a minutar guardados no banco de dados.")

processos a minutar guardados no banco de dados.


In [ ]:
#EXECUTA! Pega e resume as últimas petições
for processo in df_pendentes:
    eproc.entrar_no_processo(navegador, processo)

    with sqlite3.connect("urcaciv.db") as conn:
        texto_peticao = pega_ultima_peticao(navegador)

        print("========== RESUMO ==========")
        print(texto_peticao)


        with closing(conn.cursor()) as cursor:
            cursor.execute(
                f"UPDATE {perfil} SET pet = ? WHERE num_processo = ?",    
                (texto_peticao, processo)
            )
            conn.commit()
        
        resumo_ollama = ollama_resumo(texto_peticao)
        print("========== RESUMO ==========")
        print(resumo_ollama)
        print("============================")

        with closing(conn.cursor()) as cursor:
            cursor.execute(
                f"UPDATE {perfil} SET resumo = ? WHERE num_processo = ?",    
                (resumo_ollama, processo)
            )
            conn.commit()



Página do processo 5004481-43.2023.8.21.0069 carregada com sucesso.
========== RESUMO ==========

ESTADO DO RIO GRANDE DO SUL
PREFEITURA MUNICIPAL DE BARRA FUNDA
Av. 24 de Março, 735 – Centro – Fone (54) 99655-8503 – Cep 99.585-000 – Barra Funda - RS 1
AO JUÍZO DA VARA JUDICIAL DA COMARCA DE SARANDI/RS.
EXECUÇÃO FISCAL Nº 5004481-43.2023.8.21.0069
MUNICÍPIO DE BARRA FUNDA, pessoa Jurídica de Direito Público interno, por
meio da assessora jurídica que subscreve, vem respeitosamente perante Vossa
Excelência, nos autos do processo que move em face de SUZANA ANDRADE, requerer
o que segue:
Considerando o despacho proferido sobre a aplicação do Tema 1184 de
Repercussão Geral do STF e da Resolução n. 547 do CNJ, que dispõem sobre a
extinção de execuções fiscais de baixo valor, e tendo em vista que o protesto do
título é uma condição prevista para o ajuizamento da execução fiscal e que a
Fazenda Pública não teve a oportunidade de realizá-lo, requer-se a concessão de
prazo de 180 (cento e oiten

InvalidSessionIdException: Message: invalid session id
Stacktrace:
	GetHandleVerifier [0x0x7ff7221d6f75+76917]
	GetHandleVerifier [0x0x7ff7221d6fd0+77008]
	(No symbol) [0x0x7ff721f89c1c]
	(No symbol) [0x0x7ff721fd055f]
	(No symbol) [0x0x7ff722008332]
	(No symbol) [0x0x7ff722002e53]
	(No symbol) [0x0x7ff722001f19]
	(No symbol) [0x0x7ff721f54b05]
	GetHandleVerifier [0x0x7ff7224ad2ad+3051437]
	GetHandleVerifier [0x0x7ff7224a7903+3028483]
	GetHandleVerifier [0x0x7ff7224c589d+3151261]
	GetHandleVerifier [0x0x7ff7221f183e+185662]
	GetHandleVerifier [0x0x7ff7221f96ff+218111]
	(No symbol) [0x0x7ff721f53b00]
	GetHandleVerifier [0x0x7ff7225c5f18+4201496]
	BaseThreadInitThunk [0x0x7ffda032e8d7+23]
	RtlUserThreadStart [0x0x7ffda0a7c34c+44]
